# LangGraph 企业 Agent 演示（阶段 3，Mock）
本 Notebook 演示：状态机结构、路由决策、工具调用（含故障与降级）、拒答、PostgreSQL 检查点持久化、多轮记忆、人工审批中断与恢复。
> ⚠️ 全程使用 **Mock 智能体**（规则 + 抽取式生成），**不调用任何大模型、无付费请求**。结果只说明流程与守卫是否按设计工作。

In [1]:
import os, sys, json, re, uuid
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "backend" / "src"), str(ROOT / "evals"), str(ROOT)]
os.chdir(ROOT)
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")  # 只读 OpsPilot 自己的 .env；不打印任何密钥
base = os.environ.get("OPSPILOT_TEST_DATABASE_URL") or re.sub(r"/[^/]+$", "/opspilot_test", os.environ["OPSPILOT_DATABASE_URL"])
assert base.rsplit("/", 1)[1].endswith("_test"), "只允许在 *_test 库上运行"
from opspilot.core.config import Settings
settings = Settings(database_url=base, agent_llm_provider="mock", agent_checkpoint_auto_setup=False)
assert settings.agent_llm_provider == "mock" and settings.paid_max_calls == 0
print("agent provider:", settings.agent_llm_provider, "| paid_max_calls:", settings.paid_max_calls, "| db:", base.rsplit("/", 1)[1])

agent provider: mock | paid_max_calls: 0 | db: opspilot_test


## 1. 准备：迁移、种子数据、语料（与 Agent 评测相同，幂等）

In [2]:
import run_agent_eval as ev
ev.prepare(settings)
from opspilot.agent import checkpoint
checkpoint.setup(settings)
print("prepared")

2026-11-02 17:58:19 [info     ] ingest_indexed                 chunks=3 doc=oncall-compensation-policy.md removed_old_chunks=0 version=1


2026-11-02 17:58:19 [info     ] ingest_indexed                 chunks=4 doc=db-connection-pool-tuning.md removed_old_chunks=0 version=1


2026-11-02 17:58:19 [info     ] ingest_indexed                 chunks=3 doc=deployment-rollback-guide.pdf removed_old_chunks=0 version=1


2026-11-02 17:58:19 [info     ] ingest_indexed                 chunks=4 doc=order-api-error-codes.md removed_old_chunks=0 version=1


2026-11-02 17:58:19 [info     ] ingest_indexed                 chunks=7 doc=order-service-http500-runbook.md removed_old_chunks=0 version=1


2026-11-02 17:58:19 [info     ] ingest_indexed                 chunks=3 doc=payment-gateway-integration.pdf removed_old_chunks=0 version=1


... (4 outputs left out)

2026-11-02 17:58:20 [info     ] ingest_indexed                 chunks=3 doc=monitoring-alerting-guide.pdf removed_old_chunks=0 version=1


2026-11-02 17:58:20 [info     ] ingest_indexed                 chunks=1 doc=rate-limit-429.txt removed_old_chunks=0 version=1


2026-11-02 17:58:20 [info     ] ingest_indexed                 chunks=3 doc=vendor-integration-notes.md removed_old_chunks=0 version=1


prepared


## 2. 状态机结构
节点通过 `Command(goto=...)` 路由，下图是 LangGraph 自动导出的结构（虚线 = 条件路由）；注释版见 `docs/diagrams/stage3_agent_graph.png`。

In [3]:
from opspilot.agent.graph import AgentDeps, build_graph
from opspilot.agent.llm import MockAgentLLM
from opspilot.llm.mock import MockEmbedding
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
sf = sessionmaker(create_engine(base), expire_on_commit=False)
g = build_graph(AgentDeps(settings, sf, MockEmbedding(), MockAgentLLM()))
gg = g.get_graph()
print("节点:", [n for n in gg.nodes if not n.startswith("__")])
print(gg.draw_mermaid()[:1500])

节点: ['guard_in', 'classify', 'plan', 'retrieve', 'assess', 'agent', 'collect', 'apply_approval', 'generate', 'verify', 'respond', 'refuse', 'fail_safe', 'approval', 'tools']


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__(<p>__start__</p>)
	guard_in(guard_in)
	classify(classify)
	plan(plan)
	retrieve(retrieve)
	assess(assess)
	agent(agent)
	collect(collect)
	apply_approval(apply_approval)
	generate(generate)
	verify(verify)
	respond(respond)
	refuse(refuse)
	fail_safe(fail_safe)
	approval(approval)
	tools(tools<hr/><small><em>__tools = search_knowledge, query_mock_logs, get_ticket, analyze_error_code, create_ticket_draft</em></small>)
	__end__(<p>__end__</p>)
	__start__ --> guard_in;
	agent -.-> assess;
	agent -.-> collect;
	agent -.-> tools;
	apply_approval -.-> respond;
	approval -.-> apply_approval;
	assess -.-> agent;
	assess -.-> generate;
	assess -.-> refuse;
	assess -.-> respond;
	classify -.-> plan;
	collect -.-> approval;
	collect -.-> assess;
	collect -.-> refuse;
	generate -.-> verify;
	guard_in -.-> classify;
	guard_in -.-> refuse;
	plan -.-> agent;
	plan -.-> refuse;
	plan -.-> respond;
	plan -.-> retrieve;
	refuse -.-> __e

## 3. AgentService + PostgresSaver

In [4]:
from langgraph.checkpoint.postgres import PostgresSaver
from opspilot.agent.service import AgentService
from opspilot.db.models import User
from sqlalchemy import select
pool = checkpoint.open_pool(settings)
svc = AgentService(settings, sf, MockEmbedding(), PostgresSaver(pool), llm=MockAgentLLM())
with sf() as s:
    U = {u.email.split("@")[0]: u for u in s.scalars(select(User))}
def show(v):
    print(f"status={v['status']} final={v.get('final_status')} mock={v.get('is_mock')}")
    print("trace:", " → ".join(n["node"] for n in v.get("node_trace") or []))
    for t in v.get("tools") or []:
        print(f"  tool {t.get('tool') or t.get('name')}: {t.get('status')} attempts={t.get('attempts')} dedup={t.get('deduplicated')}")
    if v.get("degraded"):
        print("degraded:", v["degraded"])
    print("answer:", (v.get("answer") or "")[:300].replace("\n", " "))
    if v.get("refusal_reason"):
        print("refusal_reason:", v["refusal_reason"])
print(sorted(U))

['admin', 'alice', 'bob']


## 4. 路由：知识库问答 / 多工具 ReAct

In [5]:
show(svc.start(U["alice"], "支付网关默认超时时间是多少"))

status=completed final=answered mock=True
trace: guard_in → classify → plan → retrieve → assess → generate → verify → respond
answer: 【Mock 智能体 · 规则模拟，未调用大模型；仅验证流程与工具，不代表真实模型推理能力】 [1] 默认超时时间为 3 秒，超时后返回ERR_PAY_TIMEOUT。 [2] 订单服务会把该错误映射为 HTTP 500（v2.3.1 起改为 503）。 [3] | ERR_PAY_TIMEOUT | 500/503 | 调用支付网关超时（默认超时 3 秒） | 客户端可在 30 秒后重试，最多 3 次 | [4] - 20:20 确认支付网关 P99 延迟从 300ms 升至 4.5 秒。


In [6]:
show(svc.start(U["alice"], "ORD-5003 是什么错误，怎么处理"))

status=completed final=answered mock=True
trace: guard_in → classify → plan → retrieve → assess → agent → tools → collect → assess → generate → verify → respond
  tool analyze_error_code: ok attempts=None dedup=False
  tool query_mock_logs: ok attempts=None dedup=False
answer: 【Mock 智能体 · 规则模拟，未调用大模型；仅验证流程与工具，不代表真实模型推理能力】 [1] 错误码 ORD-5003：所属服务 order-service；类别 5xx；可重试：是。 [2] order-service 最近 3 条日志，按错误码统计 {'ORD-5003': 3}。 [3] | ORD-5003 | 503 | 数据库连接池耗尽 | 稍后重试；值班人员按排查手册扩容 | [4] 日志出现 "Connection is not available, request timed out after 30000ms" 或错误码 ORD-5003，说明 HikariCP 连接


## 5. 证据不足 → 拒答（不编造）

In [7]:
show(svc.start(U["alice"], "支付网关的商户号是多少"))

status=refused final=refused mock=True
trace: guard_in → classify → plan → retrieve → assess → agent → tools → collect → assess → refuse
  tool search_knowledge: ok attempts=None dedup=False
answer: 无法给出可靠回答：在你有权限访问的资料和工具结果中没有足够证据。 缺少的信息：证据提到了该对象，但没有给出具体数值
refusal_reason: value_slot_empty


## 6. 权限：普通用户查看他人工单 → 服务端 ACL 返回 not_found（不泄露存在性）→ 拒答

In [8]:
show(svc.start(U["alice"], "查一下工单 TCK-0002 的状态"))

status=refused final=refused mock=True
trace: guard_in → classify → plan → agent → tools → collect → assess → refuse
  tool get_ticket: error attempts=None dedup=False
answer: 无法给出可靠回答：在你有权限访问的资料和工具结果中没有足够证据。 缺少的信息：所需工具未能取得证据：get_ticket（not_found）
refusal_reason: tool_failures:get_ticket=not_found


## 7. 工具故障与降级（故障注入，仅测试/演示）

In [9]:
fs = settings.model_copy(update={"agent_fault_injection": True})
svc_f = AgentService(fs, sf, MockEmbedding(), PostgresSaver(pool), llm=MockAgentLLM(), faults={"query_mock_logs": "timeout"})
show(svc_f.start(U["alice"], "订单服务最近的错误日志里有什么"))

status=refused final=refused mock=True
trace: guard_in → classify → plan → retrieve → assess → agent → tools → collect → assess → refuse
  tool query_mock_logs: timeout attempts=None dedup=False
answer: 无法给出可靠回答：在你有权限访问的资料和工具结果中没有足够证据。 缺少的信息：所需工具未能取得证据：query_mock_logs（timeout）
refusal_reason: required_tool_failed


## 8. 多轮记忆（同一 thread，追问用代词）

In [10]:
v1 = svc.start(U["alice"], "查一下工单 TCK-0001 的状态")
show(v1)
v2 = svc.start(U["alice"], "它现在是什么优先级", v1["thread_id"])
show(v2)
print("same thread:", v1["thread_id"] == v2["thread_id"])

status=completed final=answered mock=True
trace: guard_in → classify → plan → agent → tools → collect → assess → generate → verify → respond
  tool get_ticket: ok attempts=None dedup=False
answer: 【Mock 智能体 · 规则模拟，未调用大模型；仅验证流程与工具，不代表真实模型推理能力】 [1] 工单 TCK-0001「订单接口 500 报错，日志出现 ORD-5003」状态 open，优先级 P2，服务 order-service。10:02 起下单接口 5xx 比例升至 8%，日志大量 ORD-5003。
status=completed final=answered mock=True
trace: guard_in → classify → plan → agent → tools → collect → assess → generate → verify → respond
  tool get_ticket: ok attempts=None dedup=False
answer: 【Mock 智能体 · 规则模拟，未调用大模型；仅验证流程与工具，不代表真实模型推理能力】 [1] 工单 TCK-0001「订单接口 500 报错，日志出现 ORD-5003」状态 open，优先级 P2，服务 order-service。10:02 起下单接口 5xx 比例升至 8%，日志大量 ORD-5003。
same thread: True


## 9. 人工审批：interrupt → 管理员批准 → 从检查点恢复（模拟进程重启）

In [11]:
v = svc.start(U["alice"], "支付网关夜间超时，帮我创建一个工单，P1")
show(v)
rid = uuid.UUID(v["run_id"])
try:
    svc.approve(rid, U["alice"], "approved")
except Exception as e:
    print("alice 审批 →", type(e).__name__)
# 新建 service + 新连接池，相当于后端重启后只依赖 PostgreSQL 中的检查点
pool.close()
pool = checkpoint.open_pool(settings)
svc2 = AgentService(settings, sf, MockEmbedding(), PostgresSaver(pool), llm=MockAgentLLM())
show(svc2.approve(rid, U["admin"], "approved"))
try:
    svc2.approve(rid, U["admin"], "approved")
except Exception as e:
    print("重复审批 →", type(e).__name__)

status=awaiting_approval final=awaiting_approval mock=True
trace: guard_in → classify → plan → agent → tools → collect
  tool create_ticket_draft: ok attempts=None dedup=False
answer: 已生成待审批工单草稿 8f301f0e…，等待管理员审批。
alice 审批 → ForbiddenError
status=completed final=answered mock=True
trace: guard_in → classify → plan → agent → tools → collect → approval → apply_approval → respond
  tool create_ticket_draft: ok attempts=None dedup=False
answer: 草稿已审批通过（第三阶段仅记录审批；真实建单在第四阶段通过 MCP 执行）。
重复审批 → ConflictError


## 10. 检查点历史（PostgreSQL `langgraph` schema）

In [12]:
h = svc2.history(rid, U["admin"])
print("checkpoints in thread:", len(h))
import psycopg
with psycopg.connect(checkpoint.conninfo(settings)) as c:
    print(c.execute("select table_name from information_schema.tables where table_schema='langgraph' order by 1").fetchall())
pool.close()

checkpoints in thread: 11
[('checkpoint_blobs',), ('checkpoint_migrations',), ('checkpoint_writes',), ('checkpoints',)]


## 小结
- 所有输出均为 Mock；真实 Qwen 测试见 `docs/STAGE3_QWEN_TEST_PLAN.md`，需单独授权。